# Semantic Embedding

We build the embedding layer for the semantic search system using `all-MiniLM-L6-v2` (384 dimensions).
Each product is represented by three vectors:
- **Product embedding** — encoded from the concatenated title, brand and description
- **Review embedding** — weighted average of individual review embeddings (weight = helpful_vote + 1)
- **Combined embedding** — blended representation (α · product + (1−α) · review) used for retrieval

Input: `data/products_cleaned.csv`, `data/reviews_topN.csv`  
Output: `data/product_embeddings.npy`, `data/review_embeddings.npy`, `data/combined_embeddings.npy`, `data/embedding_index.csv`

In [ ]:
# Importing libraries
import os
import time
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

In [ ]:
# Configuration
MODEL_NAME = "all-MiniLM-L6-v2"   # 384-dim, fast on CPU
BATCH_SIZE = 64
MIN_TOKENS = 10                    # minimum review length in tokens (from EDA)
ALPHA      = 0.5                   # product vs review embedding blend weight
DATA_DIR   = "Mean-Squared-Terrors/data"
SEED       = 42

np.random.seed(SEED)

# Paths
PRODUCTS_PATH = os.path.join(DATA_DIR, "products_cleaned.csv")
REVIEWS_PATH  = os.path.join(DATA_DIR, "reviews_topN.csv")
OUT_PROD_EMB  = os.path.join(DATA_DIR, "product_embeddings.npy")
OUT_REV_EMB   = os.path.join(DATA_DIR, "review_embeddings.npy")
OUT_COMB_EMB  = os.path.join(DATA_DIR, "combined_embeddings.npy")
OUT_INDEX     = os.path.join(DATA_DIR, "embedding_index.csv")

## Loading Data

We load the product catalogue and the top-N reviews selected during ingestion, then drop reviews
shorter than 10 tokens — a threshold chosen during EDA to filter out uninformative one-liners.

In [ ]:
products = pd.read_csv(PRODUCTS_PATH)
reviews  = pd.read_csv(REVIEWS_PATH)

# Token-length filter (decision from EDA)
reviews["tok_len"] = reviews["text"].astype(str).str.split().str.len()
before  = len(reviews)
reviews = reviews[reviews["tok_len"] >= MIN_TOKENS].copy()

print(f"Products: {len(products):,}")
print(f"Reviews:  {len(reviews):,} / {before:,} kept after tok_len >= {MIN_TOKENS} ({len(reviews)/before:.1%})")

## Model Initialization

We load the sentence transformer from the local Hugging Face cache if available, otherwise it is
downloaded (~22 MB). All embeddings are produced by this single shared model instance.

In [ ]:
model = SentenceTransformer(MODEL_NAME)
print(f"Model:         {MODEL_NAME}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

## Product Embeddings

Each product's metadata (title, brand, description) was concatenated into `product_text_base` during
ingestion. We encode that field directly and normalise to unit length so that cosine similarity equals
a plain dot product at retrieval time.

In [ ]:
texts = products["product_text_base"].fillna("").tolist()

t0 = time.time()
product_embs = model.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print(f"Done — {product_embs.shape[0]:,} vectors x {product_embs.shape[1]} dims in {time.time() - t0:.1f}s")

## Review Embeddings

For each product we encode its top-N reviews individually, then collapse them into a single vector
via a weighted average where the weight of each review is `helpful_vote + 1` (the +1 prevents
zero-weight for reviews with no votes). The aggregate is re-normalised to stay on the unit sphere.

In [ ]:
dim       = model.get_sentence_embedding_dimension()
grouped   = reviews.groupby("parent_asin")
asin_list = products["parent_asin"].tolist()

review_embs = np.zeros((len(asin_list), dim), dtype=np.float32)
no_reviews  = 0

t0 = time.time()
for i, asin in enumerate(asin_list):
    if (i + 1) % 1000 == 0:
        print(f"  {i+1}/{len(asin_list)} products processed...")

    if asin not in grouped.groups:
        no_reviews += 1
        continue

    prod_revs = grouped.get_group(asin)
    rev_texts = prod_revs["text"].astype(str).tolist()
    weights   = prod_revs["helpful_vote"].values.astype(float) + 1.0

    embs = model.encode(rev_texts, normalize_embeddings=True, show_progress_bar=False)

    # Weighted average, then re-normalise
    w   = weights / weights.sum()
    agg = (embs * w[:, np.newaxis]).sum(axis=0)
    norm = np.linalg.norm(agg)
    if norm > 0:
        agg = agg / norm
    review_embs[i] = agg

print(f"Done — {len(asin_list):,} review vectors in {time.time() - t0:.1f}s")
if no_reviews:
    print(f"Warning: {no_reviews} product(s) had no reviews after filtering — zero vector fallback")

## Combining Embeddings

We blend product and review vectors with a simple weighted sum controlled by `ALPHA`. Both sides
carry equal weight (0.5) by default. The result is re-normalised so all output vectors remain on
the unit sphere and are directly comparable at retrieval time.

In [ ]:
combined_embs = ALPHA * product_embs + (1 - ALPHA) * review_embs

# Re-normalise — zero vectors (products with no reviews) keep their fallback value
norms = np.linalg.norm(combined_embs, axis=1, keepdims=True)
norms[norms == 0] = 1.0
combined_embs = combined_embs / norms

print(f"Combined embedding shape: {combined_embs.shape}")

## Saving Outputs

We persist all three embedding matrices as `.npy` arrays and an index CSV mapping each row position
to its `parent_asin` and product title — the index is required by the retrieval layer in the next step.

In [ ]:
np.save(OUT_PROD_EMB, product_embs)
np.save(OUT_REV_EMB,  review_embs)
np.save(OUT_COMB_EMB, combined_embs)

# Index CSV: row position → parent_asin + product_title
index_df = products[["parent_asin", "product_title"]].reset_index(drop=True)
index_df.to_csv(OUT_INDEX, index=False)

print(f"Saved {OUT_PROD_EMB}   {product_embs.shape}")
print(f"Saved {OUT_REV_EMB}    {review_embs.shape}")
print(f"Saved {OUT_COMB_EMB}   {combined_embs.shape}")
print(f"Saved {OUT_INDEX}      {len(index_df):,} rows")

## Sanity Check

We run three representative queries to verify that the combined embeddings surface sensible products
before moving on to the retrieval step. Scores are cosine similarities (dot products between unit vectors).

In [ ]:
test_queries = [
    "affordable moisturizer for sensitive skin",
    "electric toothbrush with long battery life",
    "vitamins for energy and immune support",
]

for query in test_queries:
    q_emb  = model.encode([query], normalize_embeddings=True)
    scores = combined_embs @ q_emb.T
    top5   = scores.flatten().argsort()[::-1][:5]

    print(f"Query: '{query}'")
    for rank, idx in enumerate(top5, 1):
        title = products.iloc[idx]["product_title"]
        if isinstance(title, str) and len(title) > 80:
            title = title[:77] + "..."
        print(f"  {rank}. [{scores[idx, 0]:.4f}]  {title}")
    print()